In [1]:
# 1) importando libs
import unicodedata
import numpy as np
import pandas as pd

In [2]:
# 2) extraindo os .csv (base tratada)
# no Colab, faça o upload dos arquivos e troque a pasta para "/content/"
pasta = "database/"
motos = pd.read_csv(pasta + "motos.csv")
clientes = pd.read_csv(pasta + "clientes.csv")
locacoes = pd.read_csv(pasta + "locacoes.csv")
manutencoes = pd.read_csv(pasta + "manutencoes.csv")
clientes.head()

,id_cliente,nome,cpf,data_nascimento,cidade,uf,telefone,email,categoria_cnh,data_cadastro
0,C0001,João Martins Ribeiro,589.188.734-81,20/04/1996,João Pessoa,PB,(83) 94072-6032,joao.martins.ribeiro919@outlook.com,A,21/07/2025
1,C0002,João Santos,NaN,NaN,Cabedelo,PB,(83) 99940-3525,joao.santos3@outlook.com,A,13/05/2024
2,C0003,Diego Carvalho Martins,764.319.634-77,13/03/1980,João Pessoa,PB,(83) 96924-6397,diego.carvalho.martins303@outlook.com,A,25/01/2025
3,C0004,Camila Machado Barbosa,198.509.428-77,02/03/1972,São Paulo,SP,(11) 95782-7758,camila.machado.barbosa990@hotmail.com,A,29/04/2025
4,C0005,Aline Nascimento Machado,348.517.557-93,14/05/1978,Cabedelo,PB,(83) 94624-4081,NaN,A,10/09/2024


In [3]:
# 3) convertendo as colunas de data (formato dd/mm/aaaa) para datetime
motos["data_aquisicao"] = pd.to_datetime(motos["data_aquisicao"], format="%d/%m/%Y")
clientes["data_cadastro"] = pd.to_datetime(clientes["data_cadastro"], format="%d/%m/%Y")
clientes["data_nascimento"] = pd.to_datetime(clientes["data_nascimento"], format="%d/%m/%Y")
for col in ["data_retirada", "data_devolucao_prevista", "data_devolucao_real"]:
    locacoes[col] = pd.to_datetime(locacoes[col], format="%d/%m/%Y")
manutencoes["data_manutencao"] = pd.to_datetime(manutencoes["data_manutencao"], format="%d/%m/%Y")

# data de referência das consultas (a mesma usada na documentação do projeto)
data_ref = pd.Timestamp("2026-11-10")

In [4]:
# 4) volumes: quantidade de linhas e colunas de cada tabela
volumes_antes = pd.DataFrame({
    "linhas": [len(motos), len(clientes), len(locacoes), len(manutencoes)],
    "colunas": [motos.shape[1], clientes.shape[1], locacoes.shape[1], manutencoes.shape[1]],
}, index=["motos", "clientes", "locacoes", "manutencoes"])
volumes_antes

,linhas,colunas
motos,120,11
clientes,700,10
locacoes,4201,15
manutencoes,1361,8


In [5]:
# 5) nulos de cada tabela (só colunas com algum nulo)
# esses nulos ficaram de propósito: não há regra segura para inventar CPF, telefone, e-mail etc.
# data_devolucao_real nula = locação cancelada ou em andamento
nulos = pd.concat({nome: df.isnull().sum() for nome, df in
                   [("motos", motos), ("clientes", clientes), ("locacoes", locacoes), ("manutencoes", manutencoes)]})
nulos[nulos > 0].rename("nulos").to_frame()

nulos
motos       placa                    5
clientes    cpf                     32
            data_nascimento         55
            telefone                53
            email                   92
            data_cadastro            4
locacoes    data_devolucao_real    208
            km_retirada             20
            km_devolucao           264
manutencoes data_manutencao          8
            km_moto                 30

In [6]:
# 6) linhas e IDs duplicados (deve dar tudo 0)
pd.DataFrame({
    "linhas duplicadas": [motos.duplicated().sum(), clientes.duplicated().sum(),
                          locacoes.duplicated().sum(), manutencoes.duplicated().sum()],
    "IDs duplicados": [motos["id_moto"].duplicated().sum(), clientes["id_cliente"].duplicated().sum(),
                       locacoes["id_locacao"].duplicated().sum(), manutencoes["id_manutencao"].duplicated().sum()],
}, index=["motos", "clientes", "locacoes", "manutencoes"])

,linhas duplicadas,IDs duplicados
motos,0,0
clientes,0,0
locacoes,0,0
manutencoes,0,0


In [7]:
# 7) categorias: cada coluna deve ter só valores padronizados (sem variações de escrita)
for df, col in [(motos, "modelo"), (motos, "filial"), (motos, "status"), (clientes, "cidade"), (clientes, "uf"),
                (clientes, "categoria_cnh"), (locacoes, "plano"), (locacoes, "forma_pagamento"),
                (locacoes, "status_pagamento"), (locacoes, "status_locacao"), (manutencoes, "tipo")]:
    print(f"{col}: {sorted(df[col].dropna().unique())}")

modelo: ['Honda Biz 125', 'Honda CG 160 Fan', 'Honda CG 160 Titan', 'Honda NXR 160 Bros', 'Honda PCX 160', 'Honda Pop 110i', 'Shineray Jet 125', 'Yamaha Factor 150', 'Yamaha Fazer 250', 'Yamaha NMAX 160']
filial: ['João Pessoa', 'São Paulo']
status: ['Alugada', 'Disponível', 'Em manutenção']
cidade: ['Bayeux', 'Cabedelo', 'Guarulhos', 'João Pessoa', 'Não informado', 'Osasco', 'Santa Rita', 'São Paulo']
uf: ['PB', 'SP']
categoria_cnh: ['A', 'AB']
plano: ['Diário', 'Mensal', 'Semanal']
forma_pagamento: ['Boleto', 'Cartão de Crédito', 'Cartão de Débito', 'Dinheiro', 'Não informado', 'Pix']
status_pagamento: ['Atrasado', 'Cancelado', 'Não informado', 'Pago', 'Pendente']
status_locacao: ['Cancelada', 'Em andamento', 'Finalizada']
tipo: ['Bateria', 'Elétrica', 'Freios', 'Kit relação', 'Não informado', 'Reparo de acidente', 'Revisão geral', 'Troca de pneus', 'Troca de óleo']


In [8]:
# 8) período coberto por cada tabela (primeira e última data)
periodo = pd.DataFrame({
    "inicio": [motos["data_aquisicao"].min(), clientes["data_cadastro"].min(),
               locacoes["data_retirada"].min(), manutencoes["data_manutencao"].min()],
    "fim": [motos["data_aquisicao"].max(), clientes["data_cadastro"].max(),
            locacoes["data_retirada"].max(), manutencoes["data_manutencao"].max()],
}, index=["motos (aquisição)", "clientes (cadastro)", "locacoes (retirada)", "manutencoes (data)"])
periodo

,inicio,fim
motos (aquisição),2023-06-19,2025-12-17
clientes (cadastro),2023-06-13,2026-10-29
locacoes (retirada),2023-06-30,2026-11-01
manutencoes (data),2023-07-09,2026-10-31


In [9]:
# 9) manutenções com data FUTURA (depois da data de referência)
# na análise inicial foram achadas 5 datas em 2027 (MT0554, MT0729, MT0732, MT0795 e MT1254),
# então a base hidratada tinha 5 erros. Não era só o ano: os IDs seguem a ordem das datas e a km da moto
# mostra quando o serviço aconteceu (ex.: MT0554 tem 1.251 km, mas a moto já tinha 14.522 km em 02/2026).
# correção: a manutenção recebe a data da manutenção com o ID anterior
manutencoes = manutencoes.sort_values("id_manutencao").reset_index(drop=True)
futura = manutencoes["data_manutencao"] > data_ref
print("Manutenções com data futura:", futura.sum())
manutencoes.loc[futura, "data_manutencao"] = manutencoes["data_manutencao"].shift(1)[futura]
print("Depois da correção:", (manutencoes["data_manutencao"] > data_ref).sum())

Manutenções com data futura: 0
Depois da correção: 0


In [10]:
# 10) datas incoerentes nas locações e nas manutenções (deve dar tudo 0)
aquisicao = motos.set_index("id_moto")["data_aquisicao"]
pd.Series({
    "devolução prevista antes da retirada": (locacoes["data_devolucao_prevista"] < locacoes["data_retirada"]).sum(),
    "devolução real antes da retirada": (locacoes["data_devolucao_real"] < locacoes["data_retirada"]).sum(),
    "locação antes da compra da moto": (locacoes["data_retirada"] < locacoes["id_moto"].map(aquisicao)).sum(),
    "manutenção antes da compra da moto": (manutencoes["data_manutencao"] < manutencoes["id_moto"].map(aquisicao)).sum(),
}, name="ocorrências").to_frame()

,ocorrências
devolução prevista antes da retirada,0
devolução real antes da retirada,0
locação antes da compra da moto,0
manutenção antes da compra da moto,0


In [11]:
# 11) clientes cadastrados duas vezes
# regra 1: mesmo CPF e mesmo nome
# regra 2: um dos cadastros está sem CPF, mas o nome e o e-mail são iguais
# (o nome é comparado sem acento e sem maiúsculas: "Ítalo Oliveira" = "Italo Oliveira")
def sem_acento(txt):
    return "".join(c for c in unicodedata.normalize("NFKD", txt) if not unicodedata.combining(c))

clientes = clientes.sort_values("id_cliente").reset_index(drop=True)
nome_chave = clientes["nome"].str.lower().map(sem_acento)

id_mais_antigo = (clientes.groupby(clientes["cpf"] + "|" + nome_chave)["id_cliente"].transform("first")
                  .fillna(clientes["id_cliente"]))                                     # regra 1
grupo_email = nome_chave + "|" + clientes["email"]
grupo_tem_sem_cpf = clientes["cpf"].isna().groupby(grupo_email).transform("any").eq(True)
primeiro_do_grupo = clientes.groupby(grupo_email)["id_cliente"].transform("first")
usa_regra_2 = clientes["email"].notna() & grupo_tem_sem_cpf & (primeiro_do_grupo < id_mais_antigo)
id_mais_antigo = id_mais_antigo.where(~usa_regra_2, primeiro_do_grupo)                 # regra 2

duplicados = dict(zip(clientes["id_cliente"], id_mais_antigo))
duplicados = {novo: antigo for novo, antigo in duplicados.items() if novo != antigo}
print("Clientes duplicados (ID novo -> ID mais antigo):", duplicados)
clientes[clientes["id_cliente"].isin(list(duplicados) + list(duplicados.values()))]

Clientes duplicados (ID novo -> ID mais antigo): {}


,id_cliente,nome,cpf,data_nascimento,cidade,uf,telefone,email,categoria_cnh,data_cadastro


In [12]:
# 12) unificando os clientes duplicados no ID mais antigo
# 12.1) o cadastro mantido herda o que só existia no cadastro novo (ex.: CPF e telefone do C0704 vão para o C0383)
for novo, antigo in duplicados.items():
    i_antigo = clientes.index[clientes["id_cliente"] == antigo][0]
    i_novo = clientes.index[clientes["id_cliente"] == novo][0]
    clientes.loc[i_antigo] = clientes.loc[i_antigo].fillna(clientes.loc[i_novo])

# 12.2) as locações do ID novo passam para o ID mais antigo
print("Locações transferidas:", locacoes["id_cliente"].isin(duplicados).sum())
locacoes["id_cliente"] = locacoes["id_cliente"].replace(duplicados)

# 12.3) o cadastro novo é removido
clientes = clientes[~clientes["id_cliente"].isin(duplicados)].reset_index(drop=True)
print("Clientes depois da unificação:", len(clientes))

Locações transferidas: 0
Clientes depois da unificação: 700


In [13]:
# 13) data de cadastro: não pode ser depois da primeira locação do cliente (importante depois da unificação)
primeira_locacao = clientes["id_cliente"].map(locacoes.groupby("id_cliente")["data_retirada"].min())
corrigir = primeira_locacao.notna() & (clientes["data_cadastro"].isna() | (clientes["data_cadastro"] > primeira_locacao))
print("Datas de cadastro corrigidas:", corrigir.sum())
clientes.loc[corrigir, "data_cadastro"] = primeira_locacao[corrigir]

Datas de cadastro corrigidas: 0


In [14]:
# 14) relação entre as tabelas: chaves estrangeiras sem correspondência (deve dar 0 órfãos)
pd.Series({
    "locacoes.id_moto sem moto em motos": (~locacoes["id_moto"].isin(motos["id_moto"])).sum(),
    "locacoes.id_cliente sem cliente em clientes": (~locacoes["id_cliente"].isin(clientes["id_cliente"])).sum(),
    "manutencoes.id_moto sem moto em motos": (~manutencoes["id_moto"].isin(motos["id_moto"])).sum(),
}, name="registros órfãos").to_frame()

,registros órfãos
locacoes.id_moto sem moto em motos,0
locacoes.id_cliente sem cliente em clientes,0
manutencoes.id_moto sem moto em motos,0


In [15]:
# 15) removendo as linhas com nulos que afetam a análise
# locação: sem km de retirada, ou finalizada sem km de devolução -> não dá para calcular km rodado nem custo por km
# manutenção: sem data ou sem km da moto -> não entra na linha do tempo nem no custo por km
# os demais nulos ficam: CPF, telefone, e-mail e placa vazios não afetam nenhum indicador, e locação
# cancelada ou em andamento não tem data de devolução real por natureza (apagar quebraria a P4)
sem_km = locacoes["km_retirada"].isna() | ((locacoes["status_locacao"] == "Finalizada") & locacoes["km_devolucao"].isna())
manut_incompleta = manutencoes["data_manutencao"].isna() | manutencoes["km_moto"].isna()
print("Locações removidas:", sem_km.sum(), "| receita dessas locações: R$",
      round(locacoes.loc[sem_km & (locacoes["status_locacao"] != "Cancelada"), "valor_total"].sum(), 2))
print("Manutenções removidas:", manut_incompleta.sum(), "| custo dessas manutenções: R$",
      round(manutencoes.loc[manut_incompleta, "custo"].sum(), 2))
locacoes = locacoes[~sem_km].reset_index(drop=True)
manutencoes = manutencoes[~manut_incompleta].reset_index(drop=True)

Locações removidas: 73 | receita dessas locações: R$ 87379.13
Manutenções removidas: 38 | custo dessas manutenções: R$ 12398.44


In [16]:
# 16) checagens finais (todas devem dar 0)
pd.Series({
    "manutenções com data futura": (manutencoes["data_manutencao"] > data_ref).sum(),
    "clientes com mesmo nome e e-mail": clientes.dropna(subset=["email"]).duplicated(["nome", "email"]).sum(),
    "CPFs repetidos": clientes["cpf"].dropna().duplicated().sum(),
    "e-mails repetidos": clientes["email"].dropna().duplicated().sum(),
    "valores negativos": int((motos["valor_aquisicao"] < 0).sum() + (locacoes["valor_total"] < 0).sum()
                             + (manutencoes["custo"] < 0).sum()),
}, name="ocorrências").to_frame()

,ocorrências
manutenções com data futura,0
clientes com mesmo nome e e-mail,0
CPFs repetidos,0
e-mails repetidos,0
valores negativos,0


In [17]:
# 17) volumes antes x depois do tratamento
volumes_depois = pd.Series({"motos": len(motos), "clientes": len(clientes),
                            "locacoes": len(locacoes), "manutencoes": len(manutencoes)})
volumes_antes[["linhas"]].assign(depois=volumes_depois)

,linhas,depois
motos,120,120
clientes,700,700
locacoes,4201,4128
manutencoes,1361,1323


In [18]:
# 18) salvando a base tratada (mesmo formato: datas em dd/mm/aaaa e números inteiros sem ".0")
def para_csv(df, colunas_inteiras):
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = df[col].dt.strftime("%d/%m/%Y")
    for col in colunas_inteiras:
        df[col] = df[col].astype("Int64")        # inteiro que aceita nulo
    return df

para_csv(motos.sort_values("id_moto"), ["ano_fabricacao", "km_atual"]).to_csv(pasta + "motos.csv", index=False)
para_csv(clientes.sort_values("id_cliente"), []).to_csv(pasta + "clientes.csv", index=False)
para_csv(locacoes.sort_values("id_locacao"), ["km_retirada", "km_devolucao"]).to_csv(pasta + "locacoes.csv", index=False)
para_csv(manutencoes.sort_values("id_manutencao"), ["km_moto", "dias_parada"]).to_csv(pasta + "manutencoes.csv", index=False)
print("Base tratada salva em", pasta)

Base tratada salva em database/
